# Prepare FineWeb-Edu on your Mac

Open this notebook in the project with your local Python 3 kernel and choose **Run All**.
Your CPU streams FineWeb-Edu, tokenizes with the TinyLlama 32k tokenizer, and saves the corpus in
**Downloads → fineweb_edu_tinyllama**. Allow about 12.7 GB for tokens plus room for caches.
After the validation documents, the progress bar shows training tokens per second and time remaining.
Keep the Mac awake while this runs. Interrupting tokenization discards the partial run.


In [ ]:
from pathlib import Path
import shutil
import sys

# Change this path if you want to use an external disk.
OUTPUT = Path.home() / "Downloads" / "fineweb_edu_tinyllama"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {OUTPUT}")
print(f"Free disk space: {shutil.disk_usage(OUTPUT.parent).free / 1e9:.1f} GB")

cwd = Path.cwd()
repo = next((p for p in (cwd, *cwd.parents) if (p / "src/base_model_research/data.py").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook inside the base-model-research project folder.")
sys.path.insert(0, str(repo / "src"))
from base_model_research.config import DEFAULT_TOKENIZER, DEFAULT_TRAINING_TARGETS
from base_model_research.data import TokenCorpus, prepare_data
from tqdm.auto import tqdm


In [ ]:
if not (OUTPUT / "manifest.json").is_file():
    print("Opening FineWeb-Edu. The first 2,048 documents go to validation.", flush=True)
    with tqdm(
        total=DEFAULT_TRAINING_TARGETS + 1, desc="Training tokens", unit="tok", unit_scale=True, mininterval=1
    ) as progress:
        prepare_data(OUTPUT, on_progress=progress.update)

train = TokenCorpus(OUTPUT, "train")
validation = TokenCorpus(OUTPUT, "validation")
manifest = train.manifest
if (
    train.total_tokens < DEFAULT_TRAINING_TARGETS + 1
    or manifest.get("dataset") != "HuggingFaceFW/fineweb-edu"
    or manifest.get("dataset_config") != "sample-100BT"
    or manifest.get("tokenizer") != DEFAULT_TOKENIZER
    or manifest.get("validation_documents") != 2048
    or not (OUTPUT / "tokenizer" / "tokenizer_config.json").is_file()
):
    raise ValueError(f"The folder at {OUTPUT} does not contain the full experiment corpus.")
print(f"Training targets: {train.total_tokens - 1:,}   Validation tokens: {validation.total_tokens:,}")
print(f"Ready to upload: {OUTPUT}")


When the cell prints **Ready to upload**, drag the entire **fineweb_edu_tinyllama** folder from Finder
into **My Drive → bmr → data** in your browser. After the upload finishes, the manifest should
be at `My Drive/bmr/data/fineweb_edu_tinyllama/manifest.json`. Continue with **02_standard_proxy_search**
in Colab.
